In [0]:

from pyspark.sql import functions as F

orders = spark.table("ecommerce.silver.orders")
fact_orders = spark.table("ecommerce.gold.fact_orders")

# Check 1: Required order IDs must not be null
null_order_ids = (
    orders.filter(F.col("order_id").isNull()).count()
)

# Check 2: Silver orders must have unique order IDs
duplicate_order_ids = (
    orders.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Check 3: Gold fact table must have unique order IDs
duplicate_gold_order_ids = (
    fact_orders.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Check 4: Every Gold order must exist in Silver
gold_without_silver = (
    fact_orders.select("order_id")
    .join(
        orders.select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .count()
)

silver_count = orders.count()
gold_count = fact_orders.count()

checks = {
    "null_silver_order_ids": null_order_ids,
    "duplicate_silver_order_ids": duplicate_order_ids,
    "duplicate_gold_order_ids": duplicate_gold_order_ids,
    "gold_orders_missing_from_silver": gold_without_silver,
}

print("Silver order count:", silver_count)
print("Gold fact order count:", gold_count)

for check_name, failures in checks.items():
    print(f"{check_name}: {failures}")

failed_checks = {
    name: count for name, count in checks.items() if count > 0
}

if failed_checks:
    raise ValueError(f"Data quality failed: {failed_checks}")

print("All critical data quality checks passed.")